In [1]:
import os
import re
import sys
import numpy as np
import pandas as pd

sys.path.append(os.path.abspath("../../")) ; from EPF import variables
sys.path.insert(0, os.path.abspath("../../../Generic-Parallel-Compute-Helper/")) ; from memory_compute import *

# Detect free RAM now, cap this kernel so it can never exhaust the machine, and
# make every parquet read/write below stream in bounded batches instead of
# loading the whole frame at once.
install_memory_guard()

REGION = variables.TARGET_REGION

# PREDISPATCH interconnector MW-flow forecast. Column
# predispatch_mwflow_{interconnector}_h{k} is the forecast flow for the k-th
# 30-min period after each timestamp (h1 ~= +30min ... h78 ~= +39h). Ex-ante
# forecasts, leakage-free by construction. Interconnector ids are the nemseer
# INTERCONNECTORID lowercased with '-' replaced by '_'.
IC_PREFIX = "predispatch_mwflow"

# Interconnectors that connect to each region (best-effort; MNSP links use
# cryptic ids -- n_q=Terranora, t_v=Basslink, v_s=Murraylink). Guarded by
# column presence, so ids absent from the data are simply skipped.
REGION_INTERCONNECTORS = {
    "nsw": ["nsw1_qld1", "vic1_nsw1", "n_q_mnsp1"],
    "qld": ["nsw1_qld1", "n_q_mnsp1"],
    "vic": ["vic1_nsw1", "v_sa", "v_s_mnsp1", "t_v_mnsp1"],
    "sa":  ["v_sa", "v_s_mnsp1"],
}

SRC_PATH = "../1_Dataset/Processed_data/6_3_predispatch_interconnectorsoln.parquet"
OUT_PATH = "../2_Features_build/Feature_data/6_3_predispatch_interconnector.parquet"


def _ic_cols(df: pd.DataFrame, ic: str) -> list:
    """Forecast-flow columns for an interconnector, ordered by horizon."""
    found = [
        (int(re.search(r"_h(\d+)$", c).group(1)), c)
        for c in df.columns
        if c.startswith(f"{IC_PREFIX}_{ic}_h")
    ]
    return [c for _, c in sorted(found)]


def _all_interconnectors(df: pd.DataFrame) -> list:
    """Distinct interconnector ids present in the data."""
    ics = set()
    for c in df.columns:
        m = re.match(rf"{IC_PREFIX}_(.+)_h\d+$", c)
        if m:
            ics.add(m.group(1))
    return sorted(ics)


[memory_guard] cap=8.10G virtual (vms 1.60G + free 6.50G) | keeping 1.5G RAM free. Over-budget cells raise MemoryError instead of crashing VS Code.


In [2]:
# The final cell streams the source in bounded row-batches straight to disk.
# Here we only pull a tiny sample so the feature functions below can be previewed
# without ever holding the full frame in memory.
sample = peek_parquet(SRC_PATH, 10)
sample.iloc[:, :6]


,predispatch_mwflow_n_q_mnsp1_h1,predispatch_mwflow_n_q_mnsp1_h2,predispatch_mwflow_n_q_mnsp1_h3,predispatch_mwflow_n_q_mnsp1_h4,predispatch_mwflow_n_q_mnsp1_h5,predispatch_mwflow_n_q_mnsp1_h6
Date,,,,,,
2018-01-01 00:00:00,-97.0,-97.0,-89.0,-73.0,-73.0,-73.0
2018-01-01 00:05:00,-97.0,-97.0,-89.0,-73.0,-73.0,-73.0
2018-01-01 00:10:00,-97.0,-97.0,-89.0,-73.0,-73.0,-73.0
2018-01-01 00:15:00,-97.0,-97.0,-89.0,-73.0,-73.0,-73.0
2018-01-01 00:20:00,-97.0,-97.0,-89.0,-73.0,-73.0,-73.0
2018-01-01 00:25:00,-97.0,-97.0,-89.0,-73.0,-73.0,-73.0
2018-01-01 00:30:00,-97.0,-89.0,-65.0,-65.0,-73.0,-73.0
2018-01-01 00:35:00,-97.0,-89.0,-65.0,-65.0,-73.0,-73.0
2018-01-01 00:40:00,-97.0,-89.0,-65.0,-65.0,-73.0,-73.0


In [3]:
def _add_interconnector_flow_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Per-interconnector forecast MW-flow summaries. Interconnector congestion
    decides whether cheap imports can relieve the target region; a forecast
    flow reversal or a link running near its limit precedes price separation.
    Near-term flow, expected 24h flow and peak utilisation are captured for
    every interconnector present. Leakage-free forecasts, used unshifted.
    Returns only the new columns to avoid copying the full base frame.
    """
    new_cols = {}
    for ic in _all_interconnectors(df):
        cols = _ic_cols(df, ic)
        if not cols:
            continue
        sub24 = df[cols[:48]]
        new_cols[f"ic_flow_{ic}_fh1"]         = df[cols[0]].astype(np.float32)
        new_cols[f"ic_flow_{ic}_fmean_24h"]   = sub24.mean(axis=1).astype(np.float32)
        new_cols[f"ic_flow_{ic}_fmaxabs_24h"] = sub24.abs().max(axis=1).astype(np.float32)
    return pd.DataFrame(new_cols, index=df.index)


_add_interconnector_flow_features(sample)[:10]


,ic_flow_n_q_mnsp1_fh1,ic_flow_n_q_mnsp1_fmean_24h,ic_flow_n_q_mnsp1_fmaxabs_24h,ic_flow_nsw1_qld1_fh1,ic_flow_nsw1_qld1_fmean_24h,ic_flow_nsw1_qld1_fmaxabs_24h,ic_flow_t_v_mnsp1_fh1,ic_flow_t_v_mnsp1_fmean_24h,ic_flow_t_v_mnsp1_fmaxabs_24h,ic_flow_v_s_mnsp1_fh1,ic_flow_v_s_mnsp1_fmean_24h,ic_flow_v_s_mnsp1_fmaxabs_24h,ic_flow_v_sa_fh1,ic_flow_v_sa_fmean_24h,ic_flow_v_sa_fmaxabs_24h,ic_flow_vic1_nsw1_fh1,ic_flow_vic1_nsw1_fmean_24h,ic_flow_vic1_nsw1_fmaxabs_24h
Date,,,,,,,,,,,,,,,,,,
2018-01-01 00:00:00,-97.0,-36.681362,97.0,-742.000000,-196.529099,794.604614,-75.043999,-322.287537,370.350708,54.0,34.662701,63.0,401.427002,288.122162,490.178436,-131.359299,317.067169,665.771484
2018-01-01 00:05:00,-97.0,-36.681362,97.0,-742.000000,-196.529099,794.604614,-75.043999,-322.287537,370.350708,54.0,34.662701,63.0,401.427002,288.122162,490.178436,-131.359299,317.067169,665.771484
2018-01-01 00:10:00,-97.0,-36.681362,97.0,-742.000000,-196.529099,794.604614,-75.043999,-322.287537,370.350708,54.0,34.662701,63.0,401.427002,288.122162,490.178436,-131.359299,317.067169,665.771484
2018-01-01 00:15:00,-97.0,-36.681362,97.0,-742.000000,-196.529099,794.604614,-75.043999,-322.287537,370.350708,54.0,34.662701,63.0,401.427002,288.122162,490.178436,-131.359299,317.067169,665.771484
2018-01-01 00:20:00,-97.0,-36.681362,97.0,-742.000000,-196.529099,794.604614,-75.043999,-322.287537,370.350708,54.0,34.662701,63.0,401.427002,288.122162,490.178436,-131.359299,317.067169,665.771484
2018-01-01 00:25:00,-97.0,-36.681362,97.0,-742.000000,-196.529099,794.604614,-75.043999,-322.287537,370.350708,54.0,34.662701,63.0,401.427002,288.122162,490.178436,-131.359299,317.067169,665.771484
2018-01-01 00:30:00,-97.0,-35.788799,97.0,-774.999512,-188.006226,774.999512,-119.777153,-335.985291,371.443665,45.0,34.980450,63.0,350.000000,288.305145,486.943054,191.000000,317.671967,658.568970
2018-01-01 00:35:00,-97.0,-35.788799,97.0,-774.999512,-188.006226,774.999512,-119.777153,-335.985291,371.443665,45.0,34.980450,63.0,350.000000,288.305145,486.943054,191.000000,317.671967,658.568970
2018-01-01 00:40:00,-97.0,-35.788799,97.0,-774.999512,-188.006226,774.999512,-119.777153,-335.985291,371.443665,45.0,34.980450,63.0,350.000000,288.305145,486.943054,191.000000,317.671967,658.568970


In [4]:
def _add_region_interconnector_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregate forecast interconnector utilisation on the links that connect to
    the target region. High combined forecast flow means the region is leaning
    heavily on its neighbours, so any limit or reversal quickly tightens local
    supply. Absolute-flow magnitudes are used (direction-convention agnostic).
    Leakage-free forecasts.
    Returns only the new columns to avoid copying the full base frame.
    """
    new_cols = {}
    ics = [ic for ic in REGION_INTERCONNECTORS.get(REGION, []) if _ic_cols(df, ic)]
    if ics:
        near = []
        mean24 = []
        for ic in ics:
            cols = _ic_cols(df, ic)
            near.append(df[cols[0]].abs())
            mean24.append(df[cols[:48]].mean(axis=1).abs())
        new_cols[f"ic_region_{REGION}_absflow_fh1"]       = pd.concat(near, axis=1).sum(axis=1).astype(np.float32)
        new_cols[f"ic_region_{REGION}_absflow_fmean_24h"] = pd.concat(mean24, axis=1).sum(axis=1).astype(np.float32)
    return pd.DataFrame(new_cols, index=df.index)


_add_region_interconnector_features(sample)[:10]


,ic_region_nsw_absflow_fh1,ic_region_nsw_absflow_fmean_24h
Date,,
2018-01-01 00:00:00,970.359314,550.277588
2018-01-01 00:05:00,970.359314,550.277588
2018-01-01 00:10:00,970.359314,550.277588
2018-01-01 00:15:00,970.359314,550.277588
2018-01-01 00:20:00,970.359314,550.277588
2018-01-01 00:25:00,970.359314,550.277588
2018-01-01 00:30:00,1062.999512,541.466980
2018-01-01 00:35:00,1062.999512,541.466980
2018-01-01 00:40:00,1062.999512,541.466980


In [5]:
def add_interconnector_features(df: pd.DataFrame) -> pd.DataFrame:
    # Both groups are row-wise over the horizon columns, so they compute
    # identically on a row-batch as on the full frame.
    return pd.concat(
        [
            _add_interconnector_flow_features(df),
            _add_region_interconnector_features(df),
        ],
        axis=1,
    )


# Retain core columns (keep_source=True): the predispatch interconnector-flow
# forecast curve is an ex-ante forecast from the run available at t ->
# leakage-free, used unshifted. Streams source + new features to disk in bounded
# batches; peak RAM is one batch, so this cannot exhaust memory.
stream_transform_parquet(SRC_PATH, OUT_PATH, transform=add_interconnector_features, keep_source=True)

import pyarrow.parquet as pq
meta = pq.ParquetFile(OUT_PATH).metadata
print("Total features:", meta.num_columns)
(meta.num_rows, meta.num_columns)


[stream] 6_3_predispatch_interconnectorsoln.parquet: 893,689 rows x 937 cols -> batches of 17,905 rows (~0.12G materialised/batch, plus pyarrow decode buffer)


Streaming..: 100%|██████████| 50/50 [00:57<00:00,  1.16s/batch]


[stream] wrote 893,689 rows -> ../2_Features_build/Feature_data/6_3_predispatch_interconnector.parquet
Total features: 957


(893689, 957)

In [ ]:
# Free this kernel's memory so the next notebook has RAM to work with
# (clears data variables + returns freed heap to the OS).
release_memory()
